# `03_instruction_dataset.ipynb` — Task C & Task D Instruction Views & Loss Masking

## 1. Designing the Model-Facing JSON Contracts
A critical design decision in LLM fine-tuning is **where to draw the boundary between neural generation and deterministic software**:
- **Task C (Document-Level Aspect + Polarity):**
  Multiple opinion annotations in the same document often share the same `(category, polarity)` (e.g., three negative mentions of `Zugfahrt`). Task C evaluates unique document-level sets. Therefore, we deduplicate and sort labels alphabetically:
  ```json
  {"labels": ["Allgemein:neutral", "Zugfahrt:negative"]}
  ```
  Note that if a document genuinely expresses **both** positive and negative sentiment toward the same category (`154` documents in `train`), both `"Zugfahrt:negative"` and `"Zugfahrt:positive"` are retained.
- **Task D (Opinion Target Extraction — Strictly No Offsets):**
  Subword tokenizers split words into variable-length character pieces; asking an autoregressive LLM to count character indices (`from=42, to=54`) causes high arithmetic error rates. Instead, the LLM outputs **verbatim target substrings** in text appearance order:
  ```json
  {"targets": [{"target": "Regionalbahn", "category": "Zugfahrt", "polarity": "negative"}]}
  ```

## 2. Why Completion-Only Loss Masking (`labels = -100`) Matters
In our Task C instruction dataset, the user prompt (guidelines + 19 category definitions + social media text) averages **~860 tokens**, whereas the target JSON completion (`{"labels":["Zugfahrt:negative"]}`) is only **~10–25 tokens**.
- If we trained with standard causal LM loss over the entire sequence, **97% of gradient updates would simply memorize our static German prompt template!**
- By setting `completion_only_loss=True` in `trl.SFTConfig` (which sets PyTorch cross-entropy target `labels = -100` for every token up to `<start_of_turn>model\n`), **100% of the gradient signal focuses on predicting the exact JSON labels.**

In [ ]:
from pathlib import Path
import os, sys, json
from IPython.display import Image, display, Markdown
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
    os.chdir(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from germeval_llm.convert import ensure_derived_datasets_ready
ensure_derived_datasets_ready()
print("Repository initialized:", ROOT.name)


In [ ]:
from germeval_llm.schemas import TaskCExample, TaskDExample
from germeval_llm.prompts import format_gemma_chat_string, compute_completion_only_labels

with Path('data/derived/task_c/dev.jsonl').open() as f:
    c_ex = TaskCExample.model_validate_json(f.readline())
with Path('data/derived/task_d/dev.jsonl').open() as f:
    d_ex = TaskDExample.model_validate_json(f.readline())

print('--- Task C Example ---')
print('Gold labels (C2):', c_ex.gold_labels)
print('Gold categories (C1):', c_ex.gold_categories)
print('Completion:', c_ex.completion[0].content)

print('\n--- Task D Example (Strictly NO Offsets) ---')
print('Completion:', d_ex.completion[0].content)
assert 'from' not in d_ex.completion[0].content and 'to' not in d_ex.completion[0].content

In [ ]:
# Verify completion-only loss masking on Task C example
full_chat = format_gemma_chat_string(c_ex.prompt, c_ex.completion)
marker = '<start_of_turn>model\n'
input_ids = [ord(ch) for ch in full_chat]
marker_ids = [ord(ch) for ch in marker]
labels = compute_completion_only_labels(input_ids, marker_ids)
masked_count = sum(1 for t in labels if t == -100)
active_count = sum(1 for t in labels if t != -100)
unmasked_text = ''.join(chr(t) for t in labels if t != -100)
print(f'Masked prompt characters: {masked_count} | Active completion characters: {active_count}')
print('Unmasked tokens contributing to SFT loss:\n', unmasked_text)
assert unmasked_text.startswith('{"labels":')

### Key Takeaways
- Both Task C and Task D share the exact same 19-category German guideline descriptions (`taxonomy.py`).
- Completion-only loss masking masks 100% of the prompt up to `<start_of_turn>model\n`, focusing all adapter capacity on the JSON output.